# Comparative and Explainable Machine Learning for Multi-Class Intrusion Detection in IoT/IIoT Networks

**Student:** Muhammad Abdullah  
**Registration ID:** 232052  
**Dataset:** TON_IoT network flows (Hugging Face)

This research-grade workflow compares Random Forest, LightGBM, XGBoost, Logistic Regression, and CatBoost for ten-class attack attribution and secondary binary detection. It audits identity leakage, evaluates compact feature sets, uncertainty, explanations, runtime, and deployable artifacts. Published values are context only; this is not claimed as an exact reproduction.

## Research questions and literature grounding

**Main question:** How can machine learning provide accurate, bias-aware, interpretable and computationally practical intrusion detection in heterogeneous IoT/IIoT traffic?

Research questions cover binary detection and multiclass attribution; identity leakage; compact features; minority-class performance and confidence; and deployment costs. The contribution is a controlled comparative framework, not a new algorithm. Shaikhanova et al. (Sensors, 2025) motivates identity hygiene, binary plus multiclass audit, per-class analysis and compute reporting. The original TON_IoT work describes a heterogeneous UNSW Canberra testbed with edge, fog and cloud components. CatBoost is included for its ordered boosting and native categorical handling, not as a novel algorithm.

References: [Primary baseline](https://doi.org/10.3390/s25247519) · [TON_IoT dataset study](https://doi.org/10.1016/j.iot.2021.100459) · [CatBoost](https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html) · [Dataset](https://huggingface.co/datasets/codymlewis/TON_IoT_network)

Scores are not targets. Dataset versions, label definitions, features and preprocessing differ, so score differences alone do not establish superiority.


## 1 — Safe environment check

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
import importlib.util, subprocess, sys, platform
optional=['datasets','lightgbm','xgboost','catboost','shap','joblib']
for p in optional:
    if importlib.util.find_spec(p) is None: subprocess.check_call([sys.executable,'-m','pip','install',p])
# Never upgrade Colab's core numpy/pandas/scipy/scikit-learn stack.
for p in ['numpy','pandas','scipy','sklearn']+optional:
    try:
        m=__import__(p); print(p,getattr(m,'__version__','available'))
    except Exception as e: print(p,'unavailable',repr(e))
print('Python',platform.python_version())


## 2 — Imports

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
import os,time,tempfile,platform,warnings
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split,StratifiedKFold,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve
import lightgbm as lgb,xgboost as xgb,catboost as cb,joblib
warnings.filterwarnings('ignore',category=FutureWarning)
SEED=42; np.random.seed(SEED); OUT=Path('.')
print('Muhammad Abdullah | 232052')


## 3 — Load TON_IoT

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
ds=load_dataset("codymlewis/TON_IoT_network")
print(ds)


## 4 — Dataframe and dictionary

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
split_name='train' if 'train' in ds else list(ds.keys())[0]
df=ds[split_name].to_pandas()
print('split',split_name,'rows',len(df),'columns',len(df.columns),'MiB',df.memory_usage(deep=True).sum()/2**20)
display(df.head())
dictionary=pd.DataFrame({'dtype':df.dtypes.astype(str),'unique':df.nunique(dropna=False),'missing':df.isna().sum(),'samples':[df[c].dropna().astype(str).head(3).tolist() for c in df]})
display(dictionary); dictionary.to_csv('data_dictionary.csv')


## 5 — Targets, duplicates, missingness

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
TARGET='type'; assert TARGET in df
display(df[TARGET].value_counts().rename('count').to_frame().assign(percent=lambda z:100*z['count']/len(df)))
if 'label' in df: print('Source binary label:',df.label.value_counts(dropna=False))
dups=int(df.duplicated().sum()); df=df.drop_duplicates().reset_index(drop=True); print('Exact duplicates removed:',dups)
display((df.isna().sum().rename('missing').to_frame().query('missing > 0')))
y=df[TARGET].astype(str)


## 6 — Identifier and leakage audit / clean features

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
tokens=['ip','session','flow_id','device_id','mac','uuid','identifier','src_port','dst_port','http_user_agent','dns_query','http_uri','ssl_subject','ssl_issuer']
sus=[c for c in df if c!=TARGET and any(t in c.lower() for t in tokens)]
audit=pd.DataFrame([{'column':c,'dtype':str(df[c].dtype),'unique':df[c].nunique(dropna=False),'cardinality_ratio':df[c].nunique(dropna=False)/len(df),'samples':df[c].dropna().astype(str).head(3).tolist()} for c in sus])
display(audit.sort_values('cardinality_ratio',ascending=False))
# Remove explicit endpoint/session/device identity. Ports and behavioral categorical fields are retained for controlled audit.
removed=[c for c in df if c!=TARGET and any(t in c.lower() for t in ['src_ip','dst_ip','source_ip','destination_ip','session_id','flow_id','device_id','mac_address','uuid'])]
X=df.drop(columns=[TARGET,'label']+removed,errors='ignore').copy()
print('Removed fields/reason:',[(c,'identity/memorization risk') for c in removed])
print('Original',len(df.columns),'remaining',X.shape[1],'feature reduction %',100*(len(df.columns)-X.shape[1])/len(df.columns))


## 7 — Feature engineering and EDA

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
def div(a,b): return pd.to_numeric(a,errors='coerce')/pd.to_numeric(b,errors='coerce').replace(0,np.nan)
def engineer(z):
    z=z.copy(); cols=set(z)
    for a,b,name in [('src_bytes','dst_bytes','byte'),('src_packets','dst_packets','packet')]:
        if a in cols and b in cols:
            z[name+'_ratio']=div(z[a],z[b]); z[name+'_imbalance']=pd.to_numeric(z[a],errors='coerce')-pd.to_numeric(z[b],errors='coerce')
    duration=next((c for c in ['duration','flow_duration','dur'] if c in cols),None)
    byt=next((c for c in ['bytes','total_bytes','sbytes'] if c in cols),None)
    pkt=next((c for c in ['packets','total_packets','spkts'] if c in cols),None)
    if duration and byt: z['bytes_per_second']=div(z[byt],z[duration])
    if duration and pkt: z['packets_per_second']=div(z[pkt],z[duration])
    if byt and pkt: z['bytes_per_packet']=div(z[byt],z[pkt])
    return z.replace([np.inf,-np.inf],np.nan)
X=engineer(X)
nums=X.select_dtypes(include=np.number).columns.tolist(); cats=[c for c in X if c not in nums]
print('numeric',nums,'categorical',cats)
corr=X[nums].corr(method='spearman').abs() if nums else pd.DataFrame()
if not corr.empty:
    display(corr.where(np.triu(np.ones(corr.shape),1).astype(bool)).stack().sort_values(ascending=False).head(30))
print('Correlation threshold 0.95 is diagnostic only; any decisions must use training rows only.')


## 8 — Stratified split and label mapping

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
le=LabelEncoder().fit(y_train); yt=le.transform(y_train); yv=le.transform(y_test); classes=le.classes_.tolist(); K=len(classes)
print(X_train.shape,X_test.shape,dict(zip(classes,range(K))))
display(pd.concat([y_train.value_counts(normalize=True).rename('train'),y_test.value_counts(normalize=True).rename('test')],axis=1))


## 9 — Preprocessing and model definitions

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
def ohe():
    try:return OneHotEncoder(handle_unknown='ignore',sparse_output=True)
    except TypeError:return OneHotEncoder(handle_unknown='ignore',sparse=True)
def prep(frame,scale=False):
    n=frame.select_dtypes(include=np.number).columns.tolist(); c=[x for x in frame if x not in n]
    ns=[('imp',SimpleImputer(strategy='median'))]+([('scale',StandardScaler(with_mean=False))] if scale else [])
    return ColumnTransformer([('num',Pipeline(ns),n),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',ohe())]),c)])
models={
'Random Forest':Pipeline([('prep',prep(X_train)),('model',RandomForestClassifier(n_estimators=200,max_depth=12,class_weight='balanced',random_state=42,n_jobs=4))]),
'LightGBM':Pipeline([('prep',prep(X_train)),('model',lgb.LGBMClassifier(objective='multiclass',n_estimators=200,learning_rate=.05,num_leaves=64,random_state=42,n_jobs=4,verbosity=-1))]),
'XGBoost':Pipeline([('prep',prep(X_train)),('model',xgb.XGBClassifier(objective='multi:softprob',num_class=K,n_estimators=200,learning_rate=.05,subsample=.8,random_state=42,n_jobs=4,eval_metric='mlogloss',tree_method='hist'))]),
'Logistic Regression':Pipeline([('prep',prep(X_train,True)),('model',LogisticRegression(C=1,penalty='l2',solver='saga',max_iter=1000,class_weight='balanced',random_state=42,n_jobs=4))])}
numcat=X_train.select_dtypes(include=np.number).columns.tolist(); catcols=[c for c in X_train if c not in numcat]
med=X_train[numcat].median(); modes={c:(X_train[c].mode().iloc[0] if X_train[c].notna().any() else '__MISSING__') for c in catcols}
def catframe(a):
    z=a.copy()
    for c in numcat:z[c]=pd.to_numeric(z[c],errors='coerce').fillna(med.get(c,0))
    for c in catcols:z[c]=z[c].fillna(modes[c]).astype(str)
    return z
cat_model=cb.CatBoostClassifier(loss_function='MultiClass',iterations=200,learning_rate=.05,depth=8,random_seed=42,verbose=False,allow_writing_files=False)
print('Baseline split protocol 80/20 stratified, seed 42; preprocessing fit on training only.')


## 10 — Controlled training-only tuning stage

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
# Small, stratified tuning subset; <=3-fold CV; test rows are not passed to search.
tune_n=min(30000,len(X_train)); ix=np.random.RandomState(42).choice(len(X_train),tune_n,replace=False)
Xt=X_train.iloc[ix]; ytt=yt[ix]; cv=StratifiedKFold(3,shuffle=True,random_state=42)
# Bounded RF search demonstrates the protocol; other published-style parameters remain predeclared baselines for compute control.
search=RandomizedSearchCV(models['Random Forest'],{'model__max_depth':[8,12,16,None],'model__min_samples_leaf':[1,2,4]},n_iter=3,scoring='f1_macro',cv=cv,random_state=42,n_jobs=4,verbose=1)
search.fit(Xt,ytt); print('RF training-only CV best params:',search.best_params_,'CV Macro-F1:',search.best_score_)
# Keep predeclared baseline configurations for fair reported comparison; full tuning of all five is an extension under a runtime budget.


## 11 — CatBoost and multiclass evaluation

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
models['CatBoost']=cat_model
def fit_eval(name,m):
    a=time.perf_counter()
    if name=='CatBoost':m.fit(catframe(X_train),yt,cat_features=catcols)
    else:m.fit(X_train,yt)
    train=time.perf_counter()-a; a=time.perf_counter()
    p=m.predict_proba(catframe(X_test) if name=='CatBoost' else X_test); pred=p.argmax(1); infer=time.perf_counter()-a
    try: auc=roc_auc_score(yv,p,multi_class='ovr',average='macro',labels=np.arange(K))
    except Exception: auc=np.nan
    ap=np.mean([average_precision_score((yv==i).astype(int),p[:,i]) for i in range(K)])
    return {'Model':name,'Accuracy':accuracy_score(yv,pred),'Macro Precision':precision_score(yv,pred,average='macro',zero_division=0),'Macro Recall':recall_score(yv,pred,average='macro',zero_division=0),'Macro-F1':f1_score(yv,pred,average='macro',zero_division=0),'Weighted-F1':f1_score(yv,pred,average='weighted',zero_division=0),'ROC-AUC':auc,'PR-AUC':ap,'Train Time (s)':train,'Inference Time (s)':infer,'ms/sample':1000*infer/len(yv)},pred,p
results={}; predictions={}; probabilities={}
for name,m in models.items():
    results[name],predictions[name],probabilities[name]=fit_eval(name,m)
table=pd.DataFrame(results.values()).sort_values('Macro-F1',ascending=False); display(table); table.to_csv('model_comparison.csv',index=False)


## 12 — Literature comparison

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
published={'Random Forest':{'Accuracy':.9897,'Macro-F1':.9681,'ROC-AUC':.9995},'LightGBM':{'Accuracy':.9903,'Macro-F1':.9694,'ROC-AUC':.9998},'XGBoost':{'Accuracy':.9902,'Macro-F1':.9680,'ROC-AUC':.9998}}
rows=[]
for m,metrics in published.items():
    for metric,value in metrics.items(): rows.append({'Model':m,'Metric':metric,'Published':value,'Our':results[m][metric],'Difference pp':100*(results[m][metric]-value)})
literature=pd.DataFrame(rows); display(literature); literature.to_csv('literature_comparison.csv',index=False)
print('Reference: Security Audit of IoT Device Networks (2025). Differences are not proof of improvement or exact reproduction.')


## 13 — Binary detection

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
binary=(y.str.lower()!='normal').astype(int)
xbtr,xbte,ybtr,ybte=train_test_split(X,binary,test_size=.2,random_state=42,stratify=binary)
binary_models={
'Random Forest':RandomForestClassifier(n_estimators=200,max_depth=12,class_weight='balanced',random_state=42,n_jobs=4),
'LightGBM':lgb.LGBMClassifier(objective='binary',n_estimators=200,learning_rate=.05,num_leaves=64,random_state=42,n_jobs=4,verbosity=-1),
'XGBoost':xgb.XGBClassifier(objective='binary:logistic',n_estimators=200,learning_rate=.05,subsample=.8,random_state=42,n_jobs=4,eval_metric='logloss',tree_method='hist'),
'Logistic Regression':LogisticRegression(C=1,solver='saga',max_iter=1000,class_weight='balanced',random_state=42,n_jobs=4),
'CatBoost':cb.CatBoostClassifier(loss_function='Logloss',iterations=200,learning_rate=.05,depth=8,random_seed=42,verbose=False,allow_writing_files=False)}
br=[]; bp={}; bprobs={}; bf={}
for name,est in binary_models.items():
    cat=name=='CatBoost'; m=est if cat else Pipeline([('prep',prep(xbtr,name=='Logistic Regression')),('model',est)])
    a=time.perf_counter(); m.fit(catframe(xbtr) if cat else xbtr,ybtr,**({'cat_features':catcols} if cat else {})); tr=time.perf_counter()-a
    pr=m.predict_proba(catframe(xbte) if cat else xbte)[:,1]; pred=(pr>=.5).astype(int); bp[name]=pred;bprobs[name]=pr;bf[name]=m
    br.append({'Model':name,'Accuracy':accuracy_score(ybte,pred),'Precision':precision_score(ybte,pred),'Recall':recall_score(ybte,pred),'F1':f1_score(ybte,pred),'ROC-AUC':roc_auc_score(ybte,pr),'PR-AUC':average_precision_score(ybte,pr),'Confusion Matrix':confusion_matrix(ybte,pred).tolist(),'Train time':tr})
binary_table=pd.DataFrame(br); display(binary_table); binary_table.to_csv('binary_model_comparison.csv',index=False)


## Unseen-source generalization stress test

Muhammad Abdullah | Registration ID: 232052. This analysis is intended to test the limits of the random split estimate.


In [ ]:
src_col=next((c for c in df.columns if c.lower() in {'src_ip','source_ip','srcip'}),None)
if src_col is None:
    print('No recognizable source address; grouped source holdout unavailable.')
else:
    from sklearn.model_selection import GroupShuffleSplit
    groups=df.loc[X.index,src_col].fillna('__MISSING__').astype(str)
    it,iv=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,groups))
    gx,gv=X.iloc[it],X.iloc[iv]; gy,gv_y=y.iloc[it],y.iloc[iv]
    enc=LabelEncoder().fit(gy); gy_i=enc.transform(gy); known=gv_y.isin(enc.classes_); unseen_rate=1-float(known.mean()); gv_i=enc.transform(gv_y[known])
    chosen=table.iloc[0]['Model']
    print('Train/test rows:',len(it),len(iv),'unique source groups:',groups.iloc[it].nunique(),groups.iloc[iv].nunique())
    if chosen=='CatBoost':
        gm=cb.CatBoostClassifier(loss_function='MultiClass',iterations=200,learning_rate=.05,depth=8,random_seed=42,verbose=False,allow_writing_files=False)
        gm.fit(catframe(gx),gy_i,cat_features=catcols); gp=gm.predict(catframe(gv.loc[known])).astype(int)
    else:
        from sklearn.base import clone
        gm=clone(models[chosen]); gm.fit(gx,gy_i); gp=gm.predict(gv.loc[known]).astype(int)
    print('Unseen source rows with unseen target labels:',unseen_rate,'| excluded from metric; random-split Macro-F1:',results[chosen]['Macro-F1'],'| unseen-source Macro-F1:',f1_score(gv_i,gp,average='macro'))
    print('A group split is a stronger shift diagnostic, not a substitute for independent site/device/time validation.')


## Binary operating-point threshold analysis

Muhammad Abdullah | Registration ID: 232052. This analysis is intended to test the limits of the random split estimate.


In [ ]:
bxtr,bxval,bytr,byval=train_test_split(xbtr,ybtr,test_size=.2,random_state=42,stratify=ybtr)
threshold_rows=[]
for n,base in binary_models.items():
    cat=n=='CatBoost'; m=base if cat else Pipeline([('prep',prep(bxtr,n=='Logistic Regression')),('model',base)])
    m.fit(catframe(bxtr) if cat else bxtr,bytr,**({'cat_features':catcols} if cat else {}))
    pv=m.predict_proba(catframe(bxval) if cat else bxval)[:,1]
    thresholds=np.linspace(.05,.95,91); scores=[f1_score(byval,pv>=t,zero_division=0) for t in thresholds]
    threshold=float(thresholds[int(np.argmax(scores))])
    threshold_rows.append({'Model':n,'Validation threshold':threshold,'Validation F1':max(scores),'Test recall':recall_score(ybte,bprobs[n]>=threshold),'Test precision':precision_score(ybte,bprobs[n]>=threshold,zero_division=0)})
threshold_table=pd.DataFrame(threshold_rows); display(threshold_table)
print('Thresholds are selected only on training validation data; report alongside default 0.50 results.')


## 14 — Confusion matrices and per-class analysis

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
per=[]
for name,pred in predictions.items():
    cm=confusion_matrix(yv,pred,labels=np.arange(K))
    fig,ax=plt.subplots(figsize=(9,7)); sns.heatmap(cm,ax=ax,cmap='Blues',xticklabels=classes,yticklabels=classes)
    ax.set(title=name+' confusion matrix',xlabel='Predicted',ylabel='Actual');plt.xticks(rotation=45,ha='right');plt.tight_layout()
    fig.savefig('confusion_matrix_'+name.lower().replace(' ','_')+'.png',dpi=160);plt.show()
    p,r,f,s=precision_recall_fscore_support(yv,pred,labels=np.arange(K),zero_division=0)
    per.extend([{'Model':name,'Class':classes[i],'Precision':p[i],'Recall':r[i],'F1':f[i],'Support':s[i]} for i in range(K)])
perclass=pd.DataFrame(per);display(perclass);perclass.to_csv('per_class_results.csv',index=False)
display(pd.Series(y).value_counts().to_frame('count').assign(ratio=lambda d:d['count']/d['count'].max()))
print('Accuracy alone can hide minority-class failures; inspect macro metrics and recall.')


## 15 — Importance and feature selection

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
def getimp(name,m):
    if name=='CatBoost': return pd.DataFrame({'feature':m.feature_names_,'importance':m.feature_importances_}).sort_values('importance',ascending=False)
    try: names=m.named_steps['prep'].get_feature_names_out()
    except: names=[str(i) for i in range(len(m.named_steps['model'].feature_importances_))]
    return pd.DataFrame({'feature':[s.split('__')[-1].split('_')[0] for s in names],'importance':m.named_steps['model'].feature_importances_}).groupby('feature',as_index=False).sum().sort_values('importance',ascending=False)
its={n:getimp(n,m) for n,m in models.items() if n!='Logistic Regression'}
feature_imp=pd.concat([t.assign(Model=n) for n,t in its.items()]);display(feature_imp.sort_values('importance',ascending=False).groupby('Model').head(20));feature_imp.to_csv('feature_importance.csv',index=False)
# MI on training only; capped sample for bounded cost.
pp=prep(X_train); transformed=pp.fit_transform(X_train); sample=min(20000,len(yt)); idx=np.random.RandomState(42).choice(len(yt),sample,replace=False)
mi=mutual_info_classif(transformed[idx],yt[idx],random_state=42);mi_table=pd.DataFrame({'feature':pp.get_feature_names_out(),'MI':mi}).sort_values('MI',ascending=False);display(mi_table.head(30))
+# Evaluate MI top-k using training-only rankings. Reduce encoded-column ranking to source columns.
+mi_table['source_feature']=mi_table['feature'].str.replace(r'^(num|cat)__','',regex=True)
+mi_columns=[]
+for f in mi_table.source_feature:
+    for original in X_train.columns:
+        if f==original or f.startswith(original+'_'):
+            if original not in mi_columns: mi_columns.append(original)
+            break
+mi_eval=[]
+for k in [5,10,20,30]:
+    cols=mi_columns[:min(k,len(mi_columns))]
+    if not cols: continue
+    mm=Pipeline([('prep',prep(X_train[cols])),('model',RandomForestClassifier(n_estimators=200,max_depth=12,class_weight='balanced',random_state=42,n_jobs=4))])
+    start=time.perf_counter();mm.fit(X_train[cols],yt);fit_s=time.perf_counter()-start; pred=mm.predict(X_test[cols])
+    mi_eval.append({'MI top-k':k,'actual source features':len(cols),'Accuracy':accuracy_score(yv,pred),'Macro-F1':f1_score(yv,pred,average='macro'),'Train time (s)':fit_s})
+mi_eval=pd.DataFrame(mi_eval); display(mi_eval)
# Ranking is prespecified from the training-fitted RF; no test metric chooses the ranking model.
besttree='Random Forest'; ranks=[c for c in its[besttree].feature if c in X_train]
fs=[]
for k in [len(X_train.columns),30,20,10]:
    cols=X_train.columns.tolist() if k>=len(X_train.columns) else ranks[:k]
    m=Pipeline([('prep',prep(X_train[cols])),('model',RandomForestClassifier(n_estimators=200,max_depth=12,class_weight='balanced',random_state=42,n_jobs=4))])
    a=time.perf_counter();m.fit(X_train[cols],yt);tr=time.perf_counter()-a; pred=m.predict(X_test[cols])
    fs.append({'features':len(cols),'Accuracy':accuracy_score(yv,pred),'Macro-F1':f1_score(yv,pred,average='macro'),'Train time':tr})
fs=pd.DataFrame(fs).drop_duplicates('features');display(fs)
print('Feature rankings and model selection must be performed within training/validation data for confirmatory estimates.')


## 16 — Identifier ablation and robustness

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
# Identifier-inclusive ablation uses same row indices; high-cardinality identity fields may inflate random-split scores.
Xa=engineer(df.drop(columns=[TARGET,'label'] if 'label' in df else [TARGET],errors='ignore'))
abl=[]
for label,aa in [('Bias-aware',X),('Identifier-inclusive',Xa)]:
    aa_tr=aa.loc[X_train.index];aa_te=aa.loc[X_test.index]
    m=Pipeline([('prep',prep(aa_tr)),('model',RandomForestClassifier(n_estimators=200,max_depth=12,class_weight='balanced',random_state=42,n_jobs=4))]);m.fit(aa_tr,yt);pr=m.predict(aa_te)
    abl.append({'Policy':label,'Features':aa.shape[1],'Accuracy':accuracy_score(yv,pr),'Macro-F1':f1_score(yv,pr,average='macro')})
display(pd.DataFrame(abl))
# Seed robustness is repeated stratified holdout for selected configuration (descriptive).
winner=table.iloc[0]['Model']; seedrows=[]
for seed in [42,52,62]:
    a,b,c,d=train_test_split(X,y,test_size=.2,random_state=seed,stratify=y); enc=LabelEncoder().fit(c); yi=enc.transform(c); yo=enc.transform(d)
    if winner=='CatBoost':
        m=cb.CatBoostClassifier(loss_function='MultiClass',iterations=200,learning_rate=.05,depth=8,random_seed=seed,verbose=False,allow_writing_files=False);m.fit(catframe(a),yi,cat_features=catcols);p=m.predict_proba(catframe(b)).argmax(1)
    else:
        from sklearn.base import clone
        m=clone(models[winner]);m.fit(a,yi);p=m.predict(b)
    seedrows.append({'seed':seed,'Accuracy':accuracy_score(yo,p),'Macro-F1':f1_score(yo,p,average='macro'),'Macro Recall':recall_score(yo,p,average='macro'),'Macro Precision':precision_score(yo,p,average='macro')})
seed_table=pd.DataFrame(seedrows);display(seed_table);display(seed_table.drop(columns='seed').agg(['mean','std']))


## 17 — Bootstrap, paired comparison, calibration

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
from sklearn.metrics import log_loss,brier_score_loss
calibration_metrics=pd.DataFrame([{'Model':n,'Log Loss':log_loss(ybte,bprobs[n]),'Brier Score':brier_score_loss(ybte,bprobs[n])} for n in bf])
display(calibration_metrics)
fig,ax=plt.subplots()
for n in bf:
    frac,mp=calibration_curve(ybte,bprobs[n],n_bins=10,strategy='quantile');ax.plot(mp,frac,marker='.',label=n)
ax.plot([0,1],[0,1],'k--');ax.set(xlabel='Mean predicted probability',ylabel='Observed attack fraction',title='Binary calibration');ax.legend();fig.savefig('calibration_curve.png',dpi=160);plt.show()
print('Paired bootstrap is descriptive; it is not proof of general superiority.')


## 18 — SHAP and efficiency

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
try:
    import shap
    chosen=besttree; model=models[chosen]; sample=X_test.sample(min(1000,len(X_test)),random_state=42)
    if chosen=='CatBoost':
        sv=shap.TreeExplainer(model).shap_values(catframe(sample)); vals=sv[0] if isinstance(sv,list) else sv; shap.summary_plot(vals,catframe(sample),show=False,max_display=20)
    else:
        trans=model.named_steps['prep'].transform(sample); names=model.named_steps['prep'].get_feature_names_out();sv=shap.TreeExplainer(model.named_steps['model']).shap_values(trans);vals=sv[0] if isinstance(sv,list) else np.asarray(sv)
        if vals.ndim==3: vals=vals.mean(axis=2)
        shap.summary_plot(vals,trans,feature_names=names,show=False,max_display=20)
    plt.tight_layout();plt.savefig('shap_summary.png',dpi=160,bbox_inches='tight');plt.show()
except Exception as e: print('Optional SHAP compatibility fallback:',repr(e))
sizes=[]
for n,m in models.items():
    f=Path(tempfile.gettempdir())/(n.replace(' ','_')+'.joblib')
    try:joblib.dump(m,f);size=f.stat().st_size/2**20;f.unlink(missing_ok=True)
    except:size=np.nan
    sizes.append({'Model':n,'Train time':results[n]['Train Time (s)'],'Inference time':results[n]['Inference Time (s)'],'ms/sample':results[n]['ms/sample'],'Size MB':size})
eff=pd.DataFrame(sizes);display(eff)
print('Edge suitability depends on measured latency, RAM, hardware and update needs; gateway/batch deployments have different budgets.')


## Validation-only final model selection

Muhammad Abdullah | Registration ID: 232052. The untouched test set is reserved for final performance reporting.


In [ ]:
# Select the final family on an internal holdout from the 80% training partition.
# Refit each predeclared model on the internal fit rows; validation labels are used only for selection.
X_fit,X_val,y_fit,y_val=train_test_split(X_train,y_train,test_size=.2,random_state=142,stratify=y_train)
enc_sel=LabelEncoder().fit(y_fit); yfit_i=enc_sel.transform(y_fit); yval_i=enc_sel.transform(y_val)
val_scores=[]
for n,base in models.items():
    if n=='CatBoost':
        vm=cb.CatBoostClassifier(loss_function='MultiClass',iterations=200,learning_rate=.05,depth=8,random_seed=42,verbose=False,allow_writing_files=False)
        vm.fit(catframe(X_fit),yfit_i,cat_features=catcols); vp=vm.predict(catframe(X_val)).astype(int)
    else:
        from sklearn.base import clone
        vm=clone(base); vm.fit(X_fit,yfit_i); vp=vm.predict(X_val).astype(int)
    val_scores.append({'Model':n,'Validation Macro-F1':f1_score(yval_i,vp,average='macro'),'Validation Macro Recall':recall_score(yval_i,vp,average='macro'),'Validation Macro Precision':precision_score(yval_i,vp,average='macro')})
validation_table=pd.DataFrame(val_scores).sort_values('Validation Macro-F1',ascending=False);display(validation_table)
selected_name=validation_table.iloc[0]['Model']
print('Selected using training-only validation:',selected_name,'| test table remains final reporting only.')


## 19 — Errors and final selection

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
name=selected_name; pred=predictions[name]; errors=np.flatnonzero(pred!=yv)
view=X_test.iloc[errors[:20]].copy();view.insert(0,'actual',le.inverse_transform(yv[errors[:20]]));view.insert(1,'predicted',le.inverse_transform(pred[errors[:20]]));display(view)
cm=confusion_matrix(yv,pred);print('Frequent confusion pairs:',sorted([(classes[i],classes[j],int(cm[i,j])) for i in range(K) for j in range(K) if i!=j],key=lambda x:x[2],reverse=True)[:10])
print('Validation-selected model:',name,'| untouched test metrics:',results[name])
# Performance/latency/recall/size remain secondary selection constraints.


## 20 — Save deployable artifacts

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
files={'Random Forest':'ton_iot_random_forest.pkl','LightGBM':'ton_iot_lightgbm.pkl','XGBoost':'ton_iot_xgboost.pkl','Logistic Regression':'ton_iot_logistic_regression.pkl','CatBoost':'ton_iot_catboost.pkl'}
for n,m in models.items():joblib.dump(m,files[n])
bundle={'model':models[name],'model_name':name,'feature_columns':X_train.columns.tolist(),'numeric_columns':numcat,'categorical_columns':catcols,'target_name':TARGET,'class_names':classes,'label_encoder':le,'preprocessing':models[name].named_steps['prep'] if name!='CatBoost' else None,'feature_engineering_rules':'engineer() creates safe rates, ratios and imbalances','removed_columns':removed,'selected_features':X_train.columns.tolist(),'split_protocol':'80/20 stratified, random_state 42','random_state':42,'training_metadata':{'dataset':'codymlewis/TON_IoT_network','split':split_name,'student':'Muhammad Abdullah','registration_id':'232052'},'evaluation_metrics':results[name],'catboost_numeric':numcat,'catboost_categorical':catcols,'catboost_medians':med,'catboost_modes':modes}
joblib.dump(bundle,'ton_iot_best_model.pkl')
loaded=joblib.load('ton_iot_best_model.pkl');sample=X_test.head(5)
pr=loaded['model'].predict_proba(catframe(sample) if name=='CatBoost' else sample)
print('Reload verified:',loaded['model_name'],loaded['label_encoder'].inverse_transform(pr.argmax(1)))


## 21 — Web-app inference example

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
def infer_record(record,bundle):
    row=engineer(pd.DataFrame([record]))
    for c in bundle['feature_columns']:
        if c not in row:row[c]=np.nan
    row=row[bundle['feature_columns']]
    if bundle['model_name']=='CatBoost':
        for c in bundle['catboost_numeric']:row[c]=pd.to_numeric(row[c],errors='coerce').fillna(bundle['catboost_medians'].get(c,0))
        for c in bundle['catboost_categorical']:row[c]=row[c].fillna(bundle['catboost_modes'][c]).astype(str)
    p=bundle['model'].predict_proba(row)[0];i=int(p.argmax())
    return {'predicted_attack_type':bundle['class_names'][i],'confidence':float(p[i]),'class_probabilities':dict(zip(bundle['class_names'],map(float,p)))}
print(infer_record(X_test.iloc[0].to_dict(),bundle))


## 22 — Export publication tables and figures

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
final=table.merge(eff,on='Model',how='left');display(final);final.to_csv('final_results_table.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5));sns.barplot(data=table,x='Model',y='Macro-F1',ax=ax);ax.tick_params(axis='x',rotation=25);fig.tight_layout();fig.savefig('model_comparison.png',dpi=160);plt.show()
# Paper comparison contains no fabricated values; CatBoost has no published comparison entry.
print('Exports: model_comparison.csv, binary_model_comparison.csv, per_class_results.csv, feature_importance.csv, literature_comparison.csv')


## 23 — Research conclusion

Muhammad Abdullah | Registration ID: 232052. This step is isolated so outputs can be recorded reproducibly.


In [ ]:
print(f"Top Macro-F1 model: {table.iloc[0].Model} ({table.iloc[0]['Macro-F1']:.4f}).")
x=results['XGBoost'];p=published['XGBoost'];print('XGBoost vs cited benchmark percentage-point deltas:',{k:100*(x[k]-p[k]) for k in p})
print('CatBoost Macro-F1:',results['CatBoost']['Macro-F1'],'Baseline Macro-F1:',{n:results[n]['Macro-F1'] for n in ['Random Forest','LightGBM','XGBoost','Logistic Regression']})
print('Feature selection results:');display(fs)
print('Most important features:',feature_imp.groupby('feature').importance.mean().nlargest(5).to_dict())
print('Difficult classes for selected model:',perclass[perclass.Model==name].nsmallest(3,'Recall')['Class'].tolist())
print('Limitations: one dataset and random flow split; potential source/capture dependence; no independent device/time/site validation; runtime is hardware-specific. Contribution is the controlled five-model, bias-aware, explainable evaluation, not a novel algorithm.')
